# Train Teacher UNet Model

This notebook trains a UNet teacher model for crack segmentation.

**Setup:** Go to `Runtime > Change runtime type` and select **T4 GPU**.

In [ ]:
!pip install -q albumentations opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

Set the path to your dataset folder in Google Drive. The dataset should have this structure:
```
SubDataset/
  train/
    images/
    masks/
  val/
    images/
    masks/
  test/
    images/
    masks/
```

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/SubDataset")  # <-- update this path
SAVE_DIR = Path("/content/drive/MyDrive/results/teacher")
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
PATIENCE = 15
TARGET_SIZE = (96, 96)

## UNet Model

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class AttentionGate(nn.Module):
    """Attention gate for skip connections — helps the model focus on crack regions."""

    def __init__(self, gate_ch: int, skip_ch: int, inter_ch: int) -> None:
        super().__init__()
        self.W_gate = nn.Conv2d(gate_ch, inter_ch, kernel_size=1, bias=False)
        self.W_skip = nn.Conv2d(skip_ch, inter_ch, kernel_size=1, bias=False)
        self.psi = nn.Sequential(
            nn.Conv2d(inter_ch, 1, kernel_size=1, bias=False),
            nn.Sigmoid(),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gate: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        g = self.W_gate(gate)
        s = self.W_skip(skip)
        attn = self.psi(self.relu(g + s))
        return skip * attn


class UNet(nn.Module):
    def __init__(
        self, in_channels: int = 1, base_filters: int = 64, dropout: float = 0.2
    ) -> None:
        super().__init__()
        f = base_filters

        self.enc1 = DoubleConv(in_channels, f, dropout=0.0)
        self.enc2 = DoubleConv(f, f * 2, dropout=dropout)
        self.enc3 = DoubleConv(f * 2, f * 4, dropout=dropout)
        self.enc4 = DoubleConv(f * 4, f * 8, dropout=dropout)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(f * 8, f * 16, dropout=dropout * 2)

        self.up4 = nn.ConvTranspose2d(f * 16, f * 8, kernel_size=2, stride=2)
        self.attn4 = AttentionGate(gate_ch=f * 8, skip_ch=f * 8, inter_ch=f * 4)
        self.dec4 = DoubleConv(f * 16, f * 8, dropout=dropout)
        self.up3 = nn.ConvTranspose2d(f * 8, f * 4, kernel_size=2, stride=2)
        self.attn3 = AttentionGate(gate_ch=f * 4, skip_ch=f * 4, inter_ch=f * 2)
        self.dec3 = DoubleConv(f * 8, f * 4, dropout=dropout)
        self.up2 = nn.ConvTranspose2d(f * 4, f * 2, kernel_size=2, stride=2)
        self.attn2 = AttentionGate(gate_ch=f * 2, skip_ch=f * 2, inter_ch=f)
        self.dec2 = DoubleConv(f * 4, f * 2, dropout=dropout)
        self.up1 = nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2)
        self.attn1 = AttentionGate(gate_ch=f, skip_ch=f, inter_ch=f // 2)
        self.dec1 = DoubleConv(f * 2, f, dropout=0.0)

        self.out = nn.Conv2d(f, 1, kernel_size=1)

        self._init_weights()

    def _init_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.ConvTranspose2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        up4 = self.up4(b)
        d4 = self.dec4(torch.cat([up4, self.attn4(up4, e4)], dim=1))
        up3 = self.up3(d4)
        d3 = self.dec3(torch.cat([up3, self.attn3(up3, e3)], dim=1))
        up2 = self.up2(d3)
        d2 = self.dec2(torch.cat([up2, self.attn2(up2, e2)], dim=1))
        up1 = self.up1(d2)
        d1 = self.dec1(torch.cat([up1, self.attn1(up1, e1)], dim=1))

        return self.out(d1)


def dice_focal_loss(
    pred_logits: torch.Tensor,
    target: torch.Tensor,
    smooth: float = 1.0,
    alpha: float = 0.8,  # Weights the positive class (cracks) higher
    gamma: float = 2.0,  # Penalizes the model more for being confident but wrong
) -> torch.Tensor:
    prediction_probabilities: torch.Tensor = torch.sigmoid(pred_logits)

    intersection: torch.Tensor = (prediction_probabilities * target).sum(dim=(2, 3))
    dice: torch.Tensor = 1 - (2 * intersection + smooth) / (
        prediction_probabilities.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + smooth
    )

    bce: torch.Tensor = F.binary_cross_entropy_with_logits(
        pred_logits, target, reduction="none"
    )
    pt: torch.Tensor = torch.exp(-bce)

    alpha_t = target * alpha + (1 - target) * (1 - alpha)
    focal_loss = (alpha_t * (1 - pt) ** gamma * bce).mean()

    return dice.mean() + focal_loss

## Dataset Preparation

In [ ]:
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset


class CrackDataset(Dataset):
    def __init__(
        self: "CrackDataset",
        image_paths: list[Path],
        mask_paths: list[Path],
        target_size: tuple[int, int],
        augment: bool = True,
    ) -> None:
        self.image_paths: list[Path] = image_paths
        self.mask_paths: list[Path] = mask_paths

        if augment:
            transforms: list = [
                A.RandomCrop(*target_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.Affine(
                    translate_percent=0.05,
                    scale=(0.9, 1.1),
                    rotate=(-15, 15),
                    p=0.5,
                    border_mode=cv2.BORDER_CONSTANT,
                ),
                A.RandomBrightnessContrast(
                    p=0.2, brightness_limit=0.2, contrast_limit=0.15
                ),
                # A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
                # A.CLAHE(p=0.3),
                # A.ElasticTransform(alpha=80, sigma=10, p=0.3),
            ]
        else:
            transforms = [
                A.Resize(*target_size),
            ]

        self.transform: A.Compose = A.Compose(
            transforms, additional_targets={"mask": "mask"}
        )

    def __len__(self: "CrackDataset") -> int:
        return len(self.image_paths)

    def __getitem__(
        self: "CrackDataset", idx: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        image = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)

        result: dict[str, np.ndarray] = self.transform(image=image, mask=mask)

        torch_image: torch.Tensor = (
            torch.tensor(result["image"], dtype=torch.float32) / 255.0
        )
        torch_mask: torch.Tensor = (
            torch.tensor(result["mask"], dtype=torch.float32) / 255.0 > 0.5
        ).float()

        torch_image = torch_image.unsqueeze(0)
        torch_mask = torch_mask.unsqueeze(0)

        return torch_image, torch_mask


def get_all_file_paths(directory: Path) -> list[Path]:
    return sorted(directory.rglob("*.*"))


def prepare_datasets(
    dataset_directory: Path, target_size: tuple[int, int]
) -> tuple[CrackDataset, CrackDataset, CrackDataset]:
    train_dataset: Path = dataset_directory / "train"
    train_image_paths: list[Path] = get_all_file_paths(train_dataset / "images")
    train_masks_paths: list[Path] = get_all_file_paths(train_dataset / "masks")

    assert len(train_image_paths) == len(train_masks_paths), (
        f"Train mismatch: {len(train_image_paths)} images vs {len(train_masks_paths)} masks"
    )

    val_dataset: Path = dataset_directory / "val"
    val_image_paths: list[Path] = get_all_file_paths(val_dataset / "images")
    val_masks_paths: list[Path] = get_all_file_paths(val_dataset / "masks")

    assert len(val_image_paths) == len(val_masks_paths), (
        f"Val mismatch: {len(val_image_paths)} images vs {len(val_masks_paths)} masks"
    )

    test_dataset: Path = dataset_directory / "test"
    test_image_paths: list[Path] = get_all_file_paths(test_dataset / "images")
    test_masks_paths: list[Path] = get_all_file_paths(test_dataset / "masks")

    assert len(test_image_paths) == len(test_masks_paths), (
        f"Test mismatch: {len(test_image_paths)} images vs {len(test_masks_paths)} masks"
    )

    return (
        CrackDataset(train_image_paths, train_masks_paths, target_size, augment=True),
        CrackDataset(val_image_paths, val_masks_paths, target_size, augment=False),
        CrackDataset(test_image_paths, test_masks_paths, target_size, augment=False),
    )

## Training

In [ ]:
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm


def plot_metrics(
    train_losses: list[float],
    val_losses: list[float],
    val_f1s: list[float],
    val_ious: list[float],
    val_precisions: list[float],
    val_recalls: list[float],
    val_accuracies: list[float],
    save_dir: Path,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    epochs: range = range(1, len(train_losses) + 1)

    axes[0, 0].plot(epochs, train_losses, label="Train Loss", color="steelblue")
    axes[0, 0].plot(epochs, val_losses, label="Val Loss", color="tomato")
    axes[0, 0].set_title("Model Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(epochs, val_f1s, label="F1 (Dice)", color="mediumseagreen")
    axes[0, 1].plot(epochs, val_ious, label="IoU", color="darkorange")
    axes[0, 1].set_title("F1 & IoU")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Score")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(epochs, val_precisions, label="Precision", color="royalblue")
    axes[1, 0].plot(epochs, val_recalls, label="Recall", color="crimson")
    axes[1, 0].set_title("Precision & Recall")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Score")
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(
        epochs, val_accuracies, label="Pixel Accuracy", color="mediumpurple"
    )
    axes[1, 1].set_title("Pixel Accuracy")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Accuracy")
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_dir / "training_metrics.png", dpi=150)
    plt.close(fig)


def save_worst_predictions(worst_samples: list, save_dir: Path) -> None:
    """Plots the top 5 worst predictions based on IoU score."""
    if not worst_samples:
        return

    fig, axes = plt.subplots(
        len(worst_samples), 3, figsize=(10, 3 * len(worst_samples))
    )

    # Handle case where there's only 1 sample
    if len(worst_samples) == 1:
        axes = [axes]

    for i, (score, img, mask, pred) in enumerate(worst_samples):
        # Move tensors to CPU and convert to numpy for plotting
        img_np = img.cpu().numpy().squeeze()
        mask_np = mask.cpu().numpy().squeeze()
        pred_np = pred.cpu().numpy().squeeze()

        axes[i][0].imshow(img_np, cmap="gray")
        axes[i][0].set_title(f"Image (IoU: {score:.4f})")
        axes[i][0].axis("off")

        axes[i][1].imshow(mask_np, cmap="gray")
        axes[i][1].set_title("True Mask")
        axes[i][1].axis("off")

        axes[i][2].imshow(pred_np, cmap="gray")
        axes[i][2].set_title("Predicted Mask")
        axes[i][2].axis("off")

    plt.tight_layout()
    # Overwrite the same image each epoch to save disk space
    plt.savefig(save_dir / "worst_validation_errors.png", dpi=150)
    plt.close(fig)


def create_dataloader(
    dataset: CrackDataset,
    batch_size: int,
    device: torch.device,
    shuffle: bool = True,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=device.type == "cuda",
        persistent_workers=True,
        prefetch_factor=2,
    )

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
save_path: Path = SAVE_DIR / "teacher_unet.pth"

device: torch.device = (
    torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"Training on: {device}")

train_dataset, val_dataset, test_dataset = prepare_datasets(DATASET_DIR, target_size=TARGET_SIZE)

train_loader: DataLoader = create_dataloader(train_dataset, BATCH_SIZE, device)
val_loader: DataLoader = create_dataloader(
    val_dataset, BATCH_SIZE, device, shuffle=False
)
test_loader: DataLoader = create_dataloader(
    test_dataset, BATCH_SIZE, device, shuffle=False
)

model: UNet = UNet(in_channels=1, base_filters=64).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=2, eta_min=1e-6
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
best_val_f1: float = 0.0
patience_counter: int = 0

train_losses: list[float] = []
val_losses: list[float] = []
val_f1s: list[float] = []
val_ious: list[float] = []
val_precisions: list[float] = []
val_recalls: list[float] = []
val_accuracies: list[float] = []

for epoch in tqdm(range(EPOCHS), desc="Model Training Time"):
    model.train()
    train_loss: float = 0.0

    for images, masks in tqdm(train_loader, desc="Training"):
        images, masks = (
            images.to(device, non_blocking=True),
            masks.to(device, non_blocking=True),
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16):
            predictions = model(images)
            loss = dice_focal_loss(predictions, masks)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()

    model.eval()
    val_loss: float = 0.0
    total_tp: float = 0.0
    total_fp: float = 0.0
    total_fn: float = 0.0
    total_correct: float = 0.0
    total_pixels: float = 0.0

    worst_samples: list = []

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validating"):
            images, masks = (
                images.to(device, non_blocking=True),
                masks.to(device, non_blocking=True),
            )

            with torch.autocast(device_type=device.type, dtype=torch.float16):
                predictions = model(images)
                val_loss += dice_focal_loss(predictions, masks).item()

            predictions_bin: torch.Tensor = (predictions > 1.0).float()

            for j in range(images.size(0)):
                img_single = images[j]
                mask_single = masks[j]
                pred_single = predictions_bin[j]

                intersection = (pred_single * mask_single).sum().item()
                union = (
                    pred_single.sum().item()
                    + mask_single.sum().item()
                    - intersection
                )

                iou_img = intersection / (union + 1e-8)

                worst_samples.append(
                    (
                        iou_img,
                        img_single.detach(),
                        mask_single.detach(),
                        pred_single.detach(),
                    )
                )

                worst_samples = sorted(worst_samples, key=lambda x: x[0])[:5]

            total_tp += (predictions_bin * masks).sum().item()
            total_fp += (predictions_bin * (1 - masks)).sum().item()
            total_fn += ((1 - predictions_bin) * masks).sum().item()
            total_correct += (predictions_bin == masks).float().sum().item()
            total_pixels += masks.numel()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    precision: float = total_tp / (total_tp + total_fp + 1e-8)
    recall: float = total_tp / (total_tp + total_fn + 1e-8)
    val_f1: float = 2 * precision * recall / (precision + recall + 1e-8)
    iou: float = total_tp / (total_tp + total_fp + total_fn + 1e-8)
    accuracy: float = total_correct / (total_pixels + 1e-8)

    scheduler.step(epoch)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_f1s.append(val_f1)
    val_ious.append(iou)
    val_precisions.append(precision)
    val_recalls.append(recall)
    val_accuracies.append(accuracy)

    print(
        f"Epoch {epoch + 1:03d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"F1: {val_f1:.4f} | "
        f"IoU: {iou:.4f} | "
        f"Prec: {precision:.4f} | "
        f"Rec: {recall:.4f} | "
        f"Acc: {accuracy:.4f}"
    )

    # Early Stopping Logic (based on F1)
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"  Saved best model → {save_path}")
    else:
        patience_counter += 1
        print(f"  EarlyStopping counter: {patience_counter} out of {PATIENCE}")

        if patience_counter >= PATIENCE:
            print(
                f"Early stopping triggered at epoch {epoch + 1}! Training halted to save resources."
            )
            break

    plot_metrics(
        train_losses,
        val_losses,
        val_f1s,
        val_ious,
        val_precisions,
        val_recalls,
        val_accuracies,
        SAVE_DIR,
    )

    save_worst_predictions(worst_samples, SAVE_DIR)

## Load Best Model

In [ ]:
model.load_state_dict(torch.load(save_path, map_location=device))
print(f"Loaded best model from {save_path}")